In [1]:
import os

BASE_DIR = "/content/resnet18_multiseed_asl"

SEEDS = [42, 123, 2026]

for seed in SEEDS:
    os.makedirs(
        os.path.join(BASE_DIR, f"seed_{seed}"),
        exist_ok=True
    )

print("Experiment directories created!")

Experiment directories created!


In [2]:
!pip install -q medmnist

In [3]:
import os

BASE_DIR = "/content/resnet18_multiseed_asl"
DATA_ROOT = "/content/medmnist_data"

SEEDS = [42, 123, 2026]

os.makedirs(BASE_DIR, exist_ok=True)
os.makedirs(DATA_ROOT, exist_ok=True)

print("Experiment directory:", BASE_DIR)

Experiment directory: /content/resnet18_multiseed_asl


In [4]:

# ============================================================
# RESNET18 - ASYMMETRIC LOSS MULTI-SEED EXPERIMENT
# Seeds: 42, 123, 2026
# ============================================================

import os
import gc
import json
import copy
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from PIL import Image
from contextlib import nullcontext
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torchvision.transforms import InterpolationMode

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score
)

from medmnist import ChestMNIST


# ============================================================
# 1. CONFIGURATION
# ============================================================

SEEDS = [42, 123, 2026]

BASE_DIR = "/content/resnet18_multiseed_asl"
DATA_ROOT = "/content/medmnist_data"

BATCH_SIZE = 32
NUM_WORKERS = 2

MAX_EPOCHS = 10
EARLY_STOPPING_PATIENCE = 3
MIN_DELTA = 0.0001

LEARNING_RATE = 0.0001
WEIGHT_DECAY = 0.0001

GAMMA_NEG = 4
GAMMA_POS = 1
CLIP = 0.05

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

USE_AMP = DEVICE.type == "cuda"

os.makedirs(BASE_DIR, exist_ok=True)
os.makedirs(DATA_ROOT, exist_ok=True)

print("Device:", DEVICE)
print("Mixed precision:", USE_AMP)


# ============================================================
# 2. SEED FUNCTION
# ============================================================

def set_seed(seed):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ============================================================
# 3. LOAD DATASETS
# ============================================================

print("\nLoading ChestMNIST...")

raw_train = ChestMNIST(
    split="train",
    download=True,
    size=128,
    root=DATA_ROOT
)

raw_val = ChestMNIST(
    split="val",
    download=True,
    size=128,
    root=DATA_ROOT
)

print("Train samples:", len(raw_train.imgs))
print("Validation samples:", len(raw_val.imgs))


# Six selected labels
SELECTED_INDICES = [0, 1, 2, 3, 4, 7]

CLASS_NAMES = [
    "Atelectasis",
    "Cardiomegaly",
    "Effusion",
    "Infiltration",
    "Mass",
    "Pneumothorax"
]

train_labels = np.asarray(
    raw_train.labels
)[:, SELECTED_INDICES].astype(np.float32)

val_labels = np.asarray(
    raw_val.labels
)[:, SELECTED_INDICES].astype(np.float32)

print("\nSelected classes:", CLASS_NAMES)


# ============================================================
# 4. DATASET AND AUGMENTATION
# ============================================================

train_transform = transforms.Compose([
    transforms.RandomAffine(
        degrees=7,
        translate=(0.03, 0.03),
        scale=(0.95, 1.05),
        interpolation=InterpolationMode.BILINEAR,
        fill=0
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5],
        std=[0.5]
    )
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5],
        std=[0.5]
    )
])


class SelectedLabelsDataset(Dataset):

    def __init__(self, base_dataset, transform=None):

        self.images = base_dataset.imgs

        self.labels = np.asarray(
            base_dataset.labels
        )[:, SELECTED_INDICES].astype(np.float32)

        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):

        image_array = np.asarray(
            self.images[index]
        )

        if image_array.ndim == 3 and image_array.shape[-1] == 1:
            image_array = image_array.squeeze(-1)

        if image_array.dtype != np.uint8:
            image_array = np.clip(
                image_array, 0, 255
            ).astype(np.uint8)

        image = Image.fromarray(
            image_array
        ).convert("L")

        if self.transform is not None:
            image = self.transform(image)

        label = torch.tensor(
            self.labels[index],
            dtype=torch.float32
        )

        return image, label


train_dataset = SelectedLabelsDataset(
    raw_train,
    train_transform
)

val_dataset = SelectedLabelsDataset(
    raw_val,
    val_transform
)


def seed_worker(worker_id):

    worker_seed = torch.initial_seed() % (2 ** 32)

    np.random.seed(worker_seed)
    random.seed(worker_seed)


def create_loaders(seed):

    generator = torch.Generator()
    generator.manual_seed(seed)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=USE_AMP,
        worker_init_fn=seed_worker,
        generator=generator,
        persistent_workers=False
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=USE_AMP,
        persistent_workers=False
    )

    return train_loader, val_loader, generator


# ============================================================
# 5. POSITIVE CLASS WEIGHTS (FOR REFERENCE)
# ============================================================

positive_counts = train_labels.sum(axis=0)
negative_counts = len(train_labels) - positive_counts

pos_weights = negative_counts / np.maximum(
    positive_counts, 1
)

pos_weights = np.clip(
    pos_weights, 1.0, 50.0
)

print("\nPositive weights:")

for name, weight in zip(CLASS_NAMES, pos_weights):
    print(f"{name}: {weight:.4f}")


# ============================================================
# 6. ASYMMETRIC LOSS
# ============================================================

class AsymmetricLoss(nn.Module):

    def __init__(
        self,
        gamma_neg=4,
        gamma_pos=1,
        clip=0.05,
        eps=1e-8
    ):

        super().__init__()

        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip
        self.eps = eps

    def forward(self, logits, targets):

        probabilities = torch.sigmoid(logits)

        prob_positive = probabilities
        prob_negative = 1.0 - probabilities

        if self.clip is not None and self.clip > 0:

            prob_negative = (
                prob_negative + self.clip
            ).clamp(max=1.0)

        loss = (
            targets * torch.log(
                prob_positive.clamp(min=self.eps)
            )
            +
            (1.0 - targets) * torch.log(
                prob_negative.clamp(min=self.eps)
            )
        )

        if self.gamma_neg > 0 or self.gamma_pos > 0:

            pt = (
                prob_positive * targets
                +
                prob_negative * (1.0 - targets)
            )

            gamma = (
                self.gamma_pos * targets
                +
                self.gamma_neg * (1.0 - targets)
            )

            weight = (1.0 - pt).pow(gamma)

            loss = loss * weight

        return -loss.mean()


# ============================================================
# 7. MODEL
# ============================================================

def create_model(seed):

    # Each seed gets a separately initialized classification head.
    set_seed(seed)

    model = models.resnet18(
        weights=models.ResNet18_Weights.DEFAULT
    )

    original_conv = model.conv1

    new_conv = nn.Conv2d(
        in_channels=1,
        out_channels=64,
        kernel_size=7,
        stride=2,
        padding=3,
        bias=False
    )

    with torch.no_grad():
        new_conv.weight.copy_(
            original_conv.weight.mean(
                dim=1,
                keepdim=True
            )
        )

    model.conv1 = new_conv

    model.fc = nn.Linear(
        model.fc.in_features,
        6
    )

    return model.to(DEVICE)


# ============================================================
# 8. METRICS
# ============================================================

def calculate_metrics(targets, probabilities):

    predictions = (
        probabilities >= 0.5
    ).astype(int)

    auc_values = []
    ap_values = []

    class_aucs = {}
    class_aps = {}

    for index, class_name in enumerate(CLASS_NAMES):

        y_true = targets[:, index]
        y_prob = probabilities[:, index]

        if len(np.unique(y_true)) == 2:
            auc = roc_auc_score(y_true, y_prob)
            auc_values.append(auc)
        else:
            auc = float("nan")

        ap = average_precision_score(
            y_true, y_prob
        )

        ap_values.append(ap)

        class_aucs[class_name] = float(auc)
        class_aps[class_name] = float(ap)

    return {
        "macro_auroc": float(np.nanmean(auc_values)),
        "macro_map": float(np.mean(ap_values)),
        "micro_f1": float(
            f1_score(
                targets,
                predictions,
                average="micro",
                zero_division=0
            )
        ),
        "macro_f1": float(
            f1_score(
                targets,
                predictions,
                average="macro",
                zero_division=0
            )
        ),
        "class_auroc": class_aucs,
        "class_ap": class_aps
    }


# ============================================================
# 9. VALIDATION
# ============================================================

def evaluate_model(model, loader, criterion):

    model.eval()

    total_loss = 0.0
    total_samples = 0

    all_logits = []
    all_targets = []

    with torch.no_grad():

        for images, targets in loader:

            images = images.to(
                DEVICE,
                non_blocking=True
            )

            targets = targets.to(
                DEVICE,
                non_blocking=True
            )

            context = (
                torch.autocast(
                    device_type="cuda",
                    dtype=torch.float16
                )
                if USE_AMP
                else nullcontext()
            )

            with context:
                logits = model(images)
                loss = criterion(logits, targets)

            batch_size = targets.size(0)

            total_loss += loss.item() * batch_size
            total_samples += batch_size

            all_logits.append(
                logits.float().cpu().numpy()
            )

            all_targets.append(
                targets.cpu().numpy()
            )

    average_loss = total_loss / total_samples

    logits = np.concatenate(all_logits, axis=0)
    targets = np.concatenate(all_targets, axis=0)

    probabilities = 1.0 / (
        1.0 + np.exp(-np.clip(logits, -50, 50))
    )

    metrics = calculate_metrics(
        targets.astype(int),
        probabilities
    )

    return average_loss, metrics, probabilities, targets


# ============================================================
# 10. SAVE HELPERS
# ============================================================

def save_json(data, path):

    temp_path = path + ".tmp"

    with open(temp_path, "w") as file:
        json.dump(
            data,
            file,
            indent=4,
            allow_nan=True
        )

    os.replace(temp_path, path)


def save_checkpoint(data, path):

    temp_path = path + ".tmp"

    torch.save(data, temp_path)

    os.replace(temp_path, path)


def save_best_model(model, path, seed, epoch, metric_name, value):

    checkpoint = {
        "model_state_dict": {
            key: value.detach().cpu().clone()
            for key, value in model.state_dict().items()
        },
        "architecture": "ResNet18",
        "num_classes": 6,
        "class_names": CLASS_NAMES,
        "selected_indices": SELECTED_INDICES,
        "input_size": 128,
        "seed": seed,
        "epoch": epoch,
        "selection_metric": metric_name,
        "selection_value": float(value),
        "loss": "AsymmetricLoss",
        "gamma_neg": GAMMA_NEG,
        "gamma_pos": GAMMA_POS,
        "clip": CLIP
    }

    save_checkpoint(checkpoint, path)


# ============================================================
# 11. RNG STATE HELPERS
# ============================================================

def get_rng_state():

    return {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
        "cuda": (
            torch.cuda.get_rng_state_all()
            if torch.cuda.is_available()
            else None
        )
    }


def restore_rng_state(state):

    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"])

    if torch.cuda.is_available() and state["cuda"] is not None:
        torch.cuda.set_rng_state_all(state["cuda"])


# ============================================================
# 12. RUN ONE SEED
# ============================================================

def run_seed(seed):

    print("\n" + "=" * 70)
    print(f"STARTING SEED: {seed}")
    print("=" * 70)

    experiment_dir = os.path.join(
        BASE_DIR,
        f"seed_{seed}"
    )

    os.makedirs(experiment_dir, exist_ok=True)

    last_checkpoint_path = os.path.join(
        experiment_dir,
        "last_checkpoint.pth"
    )

    summary_path = os.path.join(
        experiment_dir,
        "experiment_summary.json"
    )

    # Skip an already completed experiment.
    if os.path.exists(summary_path):

        with open(summary_path, "r") as file:
            existing_summary = json.load(file)

        print("This seed is already completed. Skipping.")
        return existing_summary

    set_seed(seed)

    train_loader, val_loader, train_generator = create_loaders(seed)

    model = create_model(seed)

    criterion = AsymmetricLoss(
        gamma_neg=GAMMA_NEG,
        gamma_pos=GAMMA_POS,
        clip=CLIP
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=1,
        min_lr=1e-6
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=USE_AMP
    )

    start_epoch = 1

    best_auc = -float("inf")
    best_map = -float("inf")

    best_auc_epoch = 0
    best_map_epoch = 0

    best_auc_metrics = None
    best_map_metrics = None

    best_auc_loss = None

    history = []
    patience_counter = 0

    # --------------------------------------------------------
    # RESUME FROM LAST COMPLETED EPOCH
    # --------------------------------------------------------

    if os.path.exists(last_checkpoint_path):

        print("Previous checkpoint found. Resuming...")

        checkpoint = torch.load(
            last_checkpoint_path,
            map_location=DEVICE,
            weights_only=False
        )

        model.load_state_dict(
            checkpoint["model_state_dict"]
        )

        optimizer.load_state_dict(
            checkpoint["optimizer_state_dict"]
        )

        scheduler.load_state_dict(
            checkpoint["scheduler_state_dict"]
        )

        scaler.load_state_dict(
            checkpoint["scaler_state_dict"]
        )

        start_epoch = checkpoint["epoch"] + 1

        best_auc = checkpoint["best_auc"]
        best_map = checkpoint["best_map"]

        best_auc_epoch = checkpoint["best_auc_epoch"]
        best_map_epoch = checkpoint["best_map_epoch"]

        best_auc_metrics = checkpoint["best_auc_metrics"]
        best_map_metrics = checkpoint["best_map_metrics"]

        best_auc_loss = checkpoint["best_auc_loss"]

        history = checkpoint["history"]
        patience_counter = checkpoint["patience_counter"]

        train_generator.set_state(
            checkpoint["train_generator_state"]
        )

        restore_rng_state(
            checkpoint["rng_state"]
        )

        print("Resuming from epoch:", start_epoch)

    else:
        print("Starting a new training run.")

    # --------------------------------------------------------
    # EPOCH LOOP
    # --------------------------------------------------------

    for epoch in range(start_epoch, MAX_EPOCHS + 1):

        model.train()

        running_loss = 0.0
        total_samples = 0

        for images, targets in train_loader:

            images = images.to(
                DEVICE,
                non_blocking=True
            )

            targets = targets.to(
                DEVICE,
                non_blocking=True
            )

            optimizer.zero_grad(set_to_none=True)

            context = (
                torch.autocast(
                    device_type="cuda",
                    dtype=torch.float16
                )
                if USE_AMP
                else nullcontext()
            )

            with context:

                logits = model(images)
                loss = criterion(logits, targets)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            batch_size = targets.size(0)

            running_loss += loss.item() * batch_size
            total_samples += batch_size

        train_loss = running_loss / total_samples

        val_loss, metrics, probabilities, targets = evaluate_model(
            model,
            val_loader,
            criterion
        )

        val_auc = metrics["macro_auroc"]
        val_map = metrics["macro_map"]

        current_lr = optimizer.param_groups[0]["lr"]

        scheduler.step(val_auc)

        # ----------------------------------------------------
        # BEST AUROC
        # ----------------------------------------------------

        improved_auc = val_auc > best_auc + MIN_DELTA

        if improved_auc:

            best_auc = val_auc
            best_auc_epoch = epoch

            best_auc_metrics = copy.deepcopy(metrics)
            best_auc_loss = val_loss

            save_best_model(
                model,
                os.path.join(
                    experiment_dir,
                    "best_model.pth"
                ),
                seed,
                epoch,
                "macro_auroc",
                best_auc
            )

            np.save(
                os.path.join(
                    experiment_dir,
                    "val_probabilities.npy"
                ),
                probabilities
            )

            np.save(
                os.path.join(
                    experiment_dir,
                    "val_targets.npy"
                ),
                targets
            )

        # ----------------------------------------------------
        # BEST MAP
        # ----------------------------------------------------

        improved_map = val_map > best_map + MIN_DELTA

        if improved_map:

            best_map = val_map
            best_map_epoch = epoch

            best_map_metrics = copy.deepcopy(metrics)

            save_best_model(
                model,
                os.path.join(
                    experiment_dir,
                    "best_model_by_map.pth"
                ),
                seed,
                epoch,
                "macro_map",
                best_map
            )

            np.save(
                os.path.join(
                    experiment_dir,
                    "val_probabilities_by_map.npy"
                ),
                probabilities
            )

        if improved_auc or improved_map:
            patience_counter = 0
        else:
            patience_counter += 1

        # ----------------------------------------------------
        # SAVE HISTORY
        # ----------------------------------------------------

        epoch_result = {
            "epoch": epoch,
            "train_loss": float(train_loss),
            "val_loss": float(val_loss),
            "macro_auroc": float(val_auc),
            "macro_map": float(val_map),
            "micro_f1": metrics["micro_f1"],
            "macro_f1": metrics["macro_f1"],
            "learning_rate": float(current_lr)
        }

        history.append(epoch_result)

        pd.DataFrame(history).to_csv(
            os.path.join(
                experiment_dir,
                "training_history.csv"
            ),
            index=False
        )

        save_json(
            history,
            os.path.join(
                experiment_dir,
                "training_history.json"
            )
        )

        print(
            f"Seed {seed} | "
            f"Epoch [{epoch}/{MAX_EPOCHS}] "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Macro AUROC: {val_auc:.4f} | "
            f"Macro mAP: {val_map:.4f} | "
            f"Micro F1: {metrics['micro_f1']:.4f} | "
            f"Macro F1: {metrics['macro_f1']:.4f} | "
            f"LR: {current_lr:.6f}"
        )

        # ----------------------------------------------------
        # SAVE COMPLETE RESUME CHECKPOINT
        # ----------------------------------------------------

        resume_data = {
            "seed": seed,
            "epoch": epoch,

            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict(),

            "best_auc": best_auc,
            "best_map": best_map,

            "best_auc_epoch": best_auc_epoch,
            "best_map_epoch": best_map_epoch,

            "best_auc_metrics": best_auc_metrics,
            "best_map_metrics": best_map_metrics,

            "best_auc_loss": best_auc_loss,

            "history": history,
            "patience_counter": patience_counter,

            "train_generator_state": train_generator.get_state(),
            "rng_state": get_rng_state()
        }

        save_checkpoint(
            resume_data,
            last_checkpoint_path
        )

        print("Checkpoint saved.")

        if patience_counter >= EARLY_STOPPING_PATIENCE:

            print("Early stopping triggered.")
            break

    # --------------------------------------------------------
    # FINAL SUMMARY
    # --------------------------------------------------------

    summary = {
        "seed": seed,
        "loss_function": "AsymmetricLoss",

        "best_auroc_epoch": best_auc_epoch,
        "best_macro_auroc": float(best_auc),

        "val_loss_at_best_auroc": best_auc_loss,

        "map_at_best_auroc": best_auc_metrics["macro_map"],
        "micro_f1_at_best_auroc": best_auc_metrics["micro_f1"],
        "macro_f1_at_best_auroc": best_auc_metrics["macro_f1"],

        "best_map_epoch": best_map_epoch,
        "best_macro_map": float(best_map),

        "auroc_at_best_map": best_map_metrics["macro_auroc"],

        "class_auroc": best_auc_metrics["class_auroc"],
        "class_ap": best_auc_metrics["class_ap"]
    }

    save_json(summary, summary_path)

    print("\nSeed completed:", seed)
    print("Best AUROC:", round(best_auc, 4))
    print("Best mAP:", round(best_map, 4))

    del model, optimizer, scheduler, scaler
    del train_loader, val_loader, train_generator

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return summary


# ============================================================
# 13. RUN ALL SEEDS
# ============================================================

all_summaries = []

for seed in SEEDS:

    result = run_seed(seed)

    all_summaries.append(result)

    save_json(
        all_summaries,
        os.path.join(
            BASE_DIR,
            "multiseed_progress.json"
        )
    )

print("\nAll seed runs completed!")


# ============================================================
# 14. FINAL COMPARISON
# ============================================================

comparison = pd.DataFrame(all_summaries)

columns = [
    "seed",
    "best_auroc_epoch",
    "best_macro_auroc",
    "map_at_best_auroc",
    "micro_f1_at_best_auroc",
    "macro_f1_at_best_auroc",
    "best_map_epoch",
    "best_macro_map"
]

comparison = comparison[columns]

print("\n" + "=" * 75)
print("MULTI-SEED COMPARISON")
print("=" * 75)

display(comparison)


# ============================================================
# 15. MEAN AND STANDARD DEVIATION
# ============================================================

metric_columns = [
    "best_macro_auroc",
    "map_at_best_auroc",
    "micro_f1_at_best_auroc",
    "macro_f1_at_best_auroc",
    "best_macro_map"
]

statistics = pd.DataFrame({
    "Mean": comparison[metric_columns].mean(),
    "Std": comparison[metric_columns].std(ddof=1)
})

print("\nMEAN AND STANDARD DEVIATION")
display(statistics)

comparison.to_csv(
    os.path.join(
        BASE_DIR,
        "multiseed_comparison.csv"
    ),
    index=False
)

statistics.to_csv(
    os.path.join(
        BASE_DIR,
        "multiseed_statistics.csv"
    )
)

save_json(
    {
        "seeds": SEEDS,
        "mean": comparison[metric_columns].mean().to_dict(),
        "std": comparison[metric_columns].std(ddof=1).to_dict()
    },
    os.path.join(
        BASE_DIR,
        "multiseed_statistics.json"
    )
)

print("\nAll results saved to:", BASE_DIR)


Device: cuda
Mixed precision: True

Loading ChestMNIST...


100%|██████████| 1.43G/1.43G [07:32<00:00, 3.15MB/s]


Train samples: 78468
Validation samples: 11219

Selected classes: ['Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration', 'Mass', 'Pneumothorax']

Positive weights:
Atelectasis: 8.8134
Cardiomegaly: 39.2400
Effusion: 7.4730
Infiltration: 4.6395
Mass: 18.6760
Pneumothorax: 20.1789

STARTING SEED: 42
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 160MB/s]


Starting a new training run.
Seed 42 | Epoch [1/10] Train Loss: 0.0432 | Val Loss: 0.0432 | Macro AUROC: 0.7953 | Macro mAP: 0.2763 | Micro F1: 0.3429 | Macro F1: 0.2986 | LR: 0.000100
Checkpoint saved.
Seed 42 | Epoch [2/10] Train Loss: 0.0407 | Val Loss: 0.0420 | Macro AUROC: 0.8007 | Macro mAP: 0.2964 | Micro F1: 0.3455 | Macro F1: 0.3315 | LR: 0.000100
Checkpoint saved.
Seed 42 | Epoch [3/10] Train Loss: 0.0404 | Val Loss: 0.0407 | Macro AUROC: 0.8025 | Macro mAP: 0.3082 | Micro F1: 0.3491 | Macro F1: 0.3301 | LR: 0.000100
Checkpoint saved.
Seed 42 | Epoch [4/10] Train Loss: 0.0401 | Val Loss: 0.0406 | Macro AUROC: 0.8129 | Macro mAP: 0.3223 | Micro F1: 0.3670 | Macro F1: 0.3360 | LR: 0.000100
Checkpoint saved.
Seed 42 | Epoch [5/10] Train Loss: 0.0398 | Val Loss: 0.0400 | Macro AUROC: 0.8114 | Macro mAP: 0.3147 | Micro F1: 0.3729 | Macro F1: 0.3563 | LR: 0.000100
Checkpoint saved.
Seed 42 | Epoch [6/10] Train Loss: 0.0396 | Val Loss: 0.0405 | Macro AUROC: 0.8111 | Macro mAP: 0.319

,seed,best_auroc_epoch,best_macro_auroc,map_at_best_auroc,micro_f1_at_best_auroc,macro_f1_at_best_auroc,best_map_epoch,best_macro_map
0,42,8,0.828281,0.342641,0.394540,0.388202,7,0.342910
1,123,10,0.820927,0.332186,0.386983,0.376030,10,0.332186
2,2026,10,0.824074,0.334358,0.388070,0.381043,10,0.334358



MEAN AND STANDARD DEVIATION


,Mean,Std
best_macro_auroc,0.824427,0.003689
map_at_best_auroc,0.336395,0.005517
micro_f1_at_best_auroc,0.389864,0.004085
macro_f1_at_best_auroc,0.381759,0.006118
best_macro_map,0.336485,0.005669



All results saved to: /content/resnet18_multiseed_asl


In [5]:
import os

for root, dirs, files in os.walk("/content/resnet18_multiseed_asl"):
    level = root.replace("/content/resnet18_multiseed_asl", "").count(os.sep)
    indent = " " * 4 * level
    print(f"{indent}{os.path.basename(root)}/")

    for file in files:
        print(f"{indent}    {file}")

resnet18_multiseed_asl/
    multiseed_statistics.csv
    multiseed_comparison.csv
    multiseed_statistics.json
    multiseed_progress.json
    seed_123/
        last_checkpoint.pth
        val_probabilities.npy
        training_history.json
        val_probabilities_by_map.npy
        best_model_by_map.pth
        val_targets.npy
        training_history.csv
        best_model.pth
        experiment_summary.json
    seed_42/
        last_checkpoint.pth
        val_probabilities.npy
        training_history.json
        val_probabilities_by_map.npy
        best_model_by_map.pth
        val_targets.npy
        training_history.csv
        best_model.pth
        experiment_summary.json
    seed_2026/
        last_checkpoint.pth
        val_probabilities.npy
        training_history.json
        val_probabilities_by_map.npy
        best_model_by_map.pth
        val_targets.npy
        training_history.csv
        best_model.pth
        experiment_summary.json


In [6]:
import shutil
import os

source_dir = "/content/resnet18_multiseed_asl"
zip_path = "/content/resnet18_multiseed_asl_backup"

shutil.make_archive(
    zip_path,
    "zip",
    source_dir
)

final_path = zip_path + ".zip"

print("ZIP created successfully!")
print(f"File size: {os.path.getsize(final_path) / (1024**2):.2f} MB")

ZIP created successfully!
File size: 395.19 MB


In [7]:
from google.colab import files

files.download("/content/resnet18_multiseed_asl_backup.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>